<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- TODO: compare dueling dqn vs dqn
- TODO: add noisy nets
- TODO: add multi-step dqn

# DQN with PyTorch Lightning

This notebook trains a **Deep Q-Network (DQN)** agent on the classic environment using **PyTorch Lightning**.

In [ ]:
!pip install gymnasium[classic-control] pytorch-lightning tianshou wandb[media] tsilva_notebook_utils==0.0.57 > /dev/null

In [ ]:
import numpy as np
from tianshou.data import PrioritizedReplayBuffer, Batch

def test_prioritized_buffer_v1(
    size: int = 10,
    hi_priority_idx: int = 3,
    n_draws: int = 20_000,
    alpha: float = 0.6,
    beta: float = 0.4,
    seed: int = 42,
) -> None:
    """Quick sanity-check that PER sampling favours high-priority items."""
    # --- deterministic rng for the whole test ---
    np.random.seed(seed)

    buf = PrioritizedReplayBuffer(size=size, alpha=alpha, beta=beta)

    # populate with dummy transitions (v1.x requires terminated & truncated)
    for i in range(size):
        buf.add(
            Batch(
                obs=i,
                act=0,
                rew=0.0,
                terminated=False,
                truncated=False,
                obs_next=i,
                info={},
            )
        )

    # make one slot much more important
    priorities = np.ones(size, dtype=np.float32)
    priorities[hi_priority_idx] = 100.0
    buf.update_weight(np.arange(size), priorities)

    # draw a lot of samples
    counts = np.zeros(size, dtype=int)
    for _ in range(n_draws):
        _, idx = buf.sample(1)          # <- no rng kwarg
        counts[idx[0]] += 1

    # analyse frequencies
    hi_freq     = counts[hi_priority_idx] / n_draws
    lo_freq_avg = counts[np.arange(size) != hi_priority_idx].mean() / n_draws
    ratio       = hi_freq / (lo_freq_avg + 1e-12)

    print(f"High-priority index {hi_priority_idx}: "
          f"{hi_freq:.3%} vs {lo_freq_avg:.3%}  (≈{ratio:.1f}×)")

    assert ratio > 5, "Priority sampling looks wrong."
    print("✔ Buffer prioritisation behaves as expected!")

# quick run
test_prioritized_buffer_v1()


🔑 Loading API keys and authentication tokens from Colab secrets:

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

# TODO: move to tsilva_notebook_utils
def load_secrets_into_env(keys):
    import os
    from dotenv import load_dotenv
    load_dotenv(override=True)

    try:
        from google.colab import userdata
        for key in keys:
            value = userdata.get(key)
            assert value, f"Key {key} not found in userdata"
            os.environ[key] = value
    except:
        from dotenv import load_dotenv
        load_dotenv(override=True)

    values = []
    for key in keys:
        value = os.getenv(key)
        assert value, f"Key {key} not found in environment variables"
        values.append(value)


_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define configuration:

In [ ]:
import os
import random, math
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import pytorch_lightning as pl
from collections import deque
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config() -> dict:
    """
    Returns a config dictionary of Double DQN hyperparameters optimized for
    specific Gymnasium environments (e.g., MountainCar-v0, CartPole-v1).
    """

    # Common config values shared across all environments
    env_id = "CartPole-v1"
    common = dict(
        env_id=env_id,
        seed=42,                         # Random seed for reproducibility
        normalize_observation=True,      # Normalize observations (especially useful for continuous features)
        # TODO: softcode this
        double_dqn=True,         
        replay_size=1000,                # Size of the experience replay buffer
        replay_alpha=0.6,                # Prioritized replay buffer alpha: balances importance sampling (setting it to 0.0 makes it a regular unprioritized buffer)
        replay_beta=0.4                  # Importance sampling correction factor: starts low, increases over time
    )

    # Environment-specific configurations
    env_configs = {
        "MountainCar-v0": dict(
            hidden_sizes=(256,),         # Two-layer neural network: larger layers help with sparse reward exploration
            gamma=0.99,                  # Discount factor: prioritize future rewards slightly less (standard value)
            batch_size=64,               # Mini-batch size for training from replay buffer
            min_buffer=1000,           # Minimum experiences before training starts (warm-up for stability)
            replay_size=10_000,         # Maximum size of experience replay buffer
            lr=6e-4,                     # Learning rate: slightly lower helps avoid instability in sparse-reward tasks
            eps_start=1.0,               # Initial ε for ε-greedy policy: start fully exploratory
            eps_end=1e-4,                # Final ε: allows for minimal exploration even late in training
            eps_total_steps=30_000,             # Multiplicative decay rate per step: slow decay for long-term exploration
            target_update=1_000,         # Frequency (in steps) to copy main net → target net (delayed stability)
            target_reward=-110.0,        # Early stopping threshold for average reward (solved benchmark)
        ),

        "CartPole-v1": dict(
            hidden_sizes=(256,),         # Smaller network is sufficient for this simple environment
            gamma=0.99,                  # Discount factor: balances immediate and future rewards
            batch_size=32,               # Slightly smaller batch helps learn fast and smooth in stable envs
            min_buffer=1_000,            # Short warm-up: CartPole has frequent reward feedback
            replay_size=50_000,          # Moderate replay buffer size is enough
            lr=1e-3,                     # Slightly higher LR to accelerate learning on this easy task
            eps_start=1.0,               # Start fully exploratory
            eps_end=0.02,                # End exploration fairly early since the task is simple
            eps_total_steps=8000,             # Multiplicative ε decay: slowly anneal exploration
            target_update=200,           # More frequent updates keep value estimates more current
            target_reward=475.0,         # Average reward threshold considered as solving the task
        ),
    }

    if env_id not in env_configs:
        raise ValueError(f"Unsupported env_id: {env_id}")

    return {**common, **env_configs[env_id]}


CONFIG = setup_config()

pl.seed_everything(CONFIG['seed'], workers=True)

Login to wandb:

In [ ]:
from wandb import login
login()

In [ ]:
def make_env(env_id):
    from gymnasium.wrappers import NormalizeObservation
    env = gym.make(env_id)
    if CONFIG['normalize_observation']: env = NormalizeObservation(env)
    env.action_space.seed(CONFIG['seed'])
    env.observation_space.seed(CONFIG['seed'])
    return env

env = make_env(CONFIG['env_id'])
env

In [ ]:
env = make_env(CONFIG['env_id'])
state, _ = env.reset(seed=CONFIG['seed'])
print(state.shape, env.action_space.n)

Create replay buffer:

In [ ]:
env = gym.make(CONFIG['env_id'])
N_INPUTS = env.observation_space.shape[0]
N_OUTPUTS = int(env.action_space.n)
N_INPUTS, N_OUTPUTS

In [ ]:
state, _ = env.reset(seed=CONFIG['seed'])
state

In [ ]:
from tianshou.data import PrioritizedReplayBuffer
replay_buffer = PrioritizedReplayBuffer(size=CONFIG['replay_size'], alpha=CONFIG['replay_alpha'], beta=CONFIG['replay_beta'])
replay_buffer

In [ ]:
class DQNModel(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        input_size = N_INPUTS
        for hidden_size in CONFIG['hidden_sizes']:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.ReLU())
            input_size = hidden_size
        layers.append(nn.Linear(input_size, N_OUTPUTS))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
model = DQNModel()
model

In [ ]:
import torch.nn as nn
import torch

class DuelingDQNModel(nn.Module):
    """
    A feed-forward dueling network:
        • shared feature extractor
        • separate value (V) and advantage (A) streams
        • Q(s,a) = V(s) + A(s,a) − mean_a A(s,a)
    """
    def __init__(self, input_dim=N_INPUTS, output_dim=N_OUTPUTS, hidden_sizes=CONFIG["hidden_sizes"]):
        super().__init__()

        # --- shared feature layers ----------------------------------------
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        self.feature = nn.Sequential(*layers)

        # --- value stream --------------------------------------------------
        self.value = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, 1),
        )

        # --- advantage stream ---------------------------------------------
        self.advantage = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, output_dim),
        )

    # ---------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not torch.is_tensor(x):                       # allow NumPy inputs
            x = torch.as_tensor(x, dtype=torch.float32)
        f = self.feature(x)
        v = self.value(f)                      # shape: (B, 1)
        a = self.advantage(f)                  # shape: (B, A)
        q = v + a - a.mean(dim=1, keepdim=True)
        return q

model = DuelingDQNModel()
model

In [ ]:

eps_end = 0.1
eps_start = 1.0
eps_decay = 0.99999999999

# plot decay
import matplotlib.pyplot as plt
def plot_epsilon_decay_linear(eps_start, eps_end, total_steps):
    eps_values = []
    for step in range(total_steps):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * (step / total_steps))
        eps_values.append(eps)

    plt.plot(range(total_steps), eps_values)
    plt.xlabel('Steps')
    plt.ylabel('Epsilon')
    plt.title('Linear Epsilon Decay Over Time')
    plt.grid()
    plt.show()

# Example usage
plot_epsilon_decay_linear(eps_start=1.0, eps_end=0.1, total_steps=10000)

In [ ]:
from tianshou.data import Batch

class _Infinite(torch.utils.data.IterableDataset):
    def __iter__(self):
        while True:
            yield torch.tensor(0) 
infinite_data_loader = torch.utils.data.DataLoader(_Infinite(), batch_size=1)
infinite_data_loader

class DQNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.save_hyperparameters()
        self.q_model = DuelingDQNModel()
        self.target_model = DuelingDQNModel()
        self.target_model.load_state_dict(self.q_model.state_dict())
        self.env = gym.make(CONFIG['env_id'])
        self.state, _ = self.env.reset(seed=CONFIG['seed'])
        self.buffer = PrioritizedReplayBuffer(size=CONFIG['replay_size'], alpha=CONFIG['replay_alpha'], beta=CONFIG['replay_beta'])
        self.episode = 0
        self.total_steps = 0
        self.episode_steps = 0
        self.episode_reward = 0
        self.episode_shaped_reward = 0
        self.episode_rewards = []  # Track all episode rewards
        self.state_counts = {}

    def forward(self, x):
        return self.q_model(x)

    @property
    def current_eps(self):
        eps_end = CONFIG['eps_end']
        eps_start = CONFIG['eps_start']
        eps_total_steps = CONFIG['eps_total_steps']
        eps = max(eps_end, eps_start - (eps_start - eps_end) * (self.total_steps / eps_total_steps))
        return eps
    
    def act(self, state):
        if random.random() < self.current_eps:
            return self.env.action_space.sample()
        else:
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad(): q = self.q_model(state)
            return int(torch.argmax(q, dim=1)[0].item())

    def train_dataloader(self):
        return infinite_data_loader

    def training_step(self, batch, batch_idx):
        action = self.act(self.state)
        next_state, reward, terminated, truncated, info = self.env.step(action)
        shaped_reward = reward
        #shaped_reward = 100 if next_state[0] >= 0.5 else reward

        # Count-based curiosity reward
        #shaped_reward = reward
        #pos, vel          = next_state
        #state_key         = (round(pos, 2),     # → coarse ~0.01 m
        #                    round(vel, 3))     # → coarse ~0.001 m/s
        #count = self.state_counts.get(state_key, 0)
        #count += 1
        #self.state_counts[state_key] = count
        #eta0, decay       = 0.02, 20_000
        #eta               = eta0 * max(0, 1 - self.total_steps / decay)
        #intrinsic         = eta / math.sqrt(count)
        #shaped_reward     = reward + intrinsic
        
        done = terminated or truncated
        self.buffer.add(Batch(
            obs=self.state,
            act=action,
            rew=shaped_reward,
            terminated=terminated,   # ← or env-specific flag
            truncated=truncated,    # ← set properly if you use a time limit
            done=done,  # Note: `done` is a combination of `terminated` and `truncated` in Tianshou
            obs_next=next_state,
            info=info
        ))

        self.state = next_state

        self.total_steps += 1
        self.episode_steps += 1
        self.episode_reward += reward
        self.episode_shaped_reward += shaped_reward

        loss = None
        if len(self.buffer) >= CONFIG['min_buffer']:
            batch, indices = self.buffer.sample(CONFIG['batch_size'])
            states = torch.tensor(batch.obs, dtype=torch.float32, device=self.device)
            actions = torch.tensor(batch.act, dtype=torch.long, device=self.device).unsqueeze(-1)
            rewards = torch.tensor(batch.rew, dtype=torch.float32, device=self.device)
            next_states = torch.tensor(batch.obs_next, dtype=torch.float32, device=self.device)
            dones = torch.tensor(batch.done, dtype=torch.float32, device=self.device)
            weights = torch.tensor(batch.weight, dtype=torch.float32, device=self.device)
            
            q_values = self.q_model(states).gather(1, actions).squeeze()
            next_q = self.target_model(next_states).max(1)[0]
            targets = rewards + CONFIG['gamma'] * next_q * (1 - dones)

            # -- compute td_errors exactly as you already do --
            td_errors = (q_values - targets.detach()).abs()

            # tiny ε so nothing gets zero prob
            priorities = (td_errors + 1e-6)

            # indices returned by buffer.sample(...) are already a NumPy array
            self.buffer.update_weight(indices, priorities)

            #td_errors = (q_values - targets.detach()).abs().detach().cpu().numpy()
            #td_errors = (q_values - targets.detach()).abs().cpu().numpy()
            #self.buffer.update(Batch(idx=indices, priority=td_errors))  # UN-comment

            #priorities = (td_errors + 1e-6).astype(np.float32)   # 1-D array, no zeros
            # make sure indices is a NumPy array too (sample already returns np.ndarray,
            # but this keeps things safe if you ever switch devices)
            #idx_np = np.asarray(indices, dtype=np.int64)

            #self.buffer.update(Batch(idx=idx_np, priority=priorities))


            # Optionally use importance-sampling weights for loss
            loss = (weights * nn.functional.mse_loss(q_values, targets.detach(), reduction='none')).mean()
            self.log('loss', loss, on_step=True, prog_bar=True)

            if self.global_step % CONFIG['target_update'] == 0:
                self.target_model.load_state_dict(self.q_model.state_dict())
                
        if done:
            #print(f"Episode {self.episode} finished after {self.episode_steps} steps with reward {self.episode_reward:.2f} (shaped: {self.episode_shaped_reward:.2f})")

            self.episode_rewards.append(self.episode_reward)
            
            # Compute stats
            rewards_arr = np.array(self.episode_rewards[-100:])  # Last 100 episodes
            min_r = float(np.min(rewards_arr))
            max_r = float(np.max(rewards_arr))
            mean_r = float(np.mean(rewards_arr))
            std_r = float(np.std(rewards_arr))

            # Log stats
            self.log('episode', self.episode, on_step=True, prog_bar=True)
            self.log('reward', self.episode_reward, on_step=True, prog_bar=True)
            self.log('shaped_reward', self.episode_shaped_reward, on_step=True, prog_bar=True)
            self.log('steps', self.episode_steps, on_step=True, prog_bar=True)
            self.log('eps', self.current_eps, on_step=True, prog_bar=True)
            self.log('reward_min', min_r, on_step=True, prog_bar=True)
            self.log('reward_max', max_r, on_step=True, prog_bar=True)
            self.log('reward_mean', mean_r, on_step=True, prog_bar=True)
            self.log('reward_std', std_r, on_step=True, prog_bar=True)

            self.episode_steps = 0
            self.episode_reward = 0
            self.episode_shaped_reward = 0
            self.episode += 1
            self.state = self.env.reset()[0]

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.q_model.parameters(), lr=CONFIG['lr'])

module = DQNModule()
module

In [ ]:
import wandb
import tempfile
import imageio
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

# Optional: set up your W&B project info
# TODO: softcode
# TODO: make sure config is logged
wandb_logger = WandbLogger(project='wip-cartpole-dqn-lightning', mode="online")


class StopOnLambda(pl.Callback):
    """
    Stop training when a user-defined lambda condition on metrics is True.
    Example:
        StopOnLambda(lambda metrics: metrics.get('reward_mean', -float('inf')) >= 475)
    """
    def __init__(self, condition, message="Stopping criterion met."):
        super().__init__()
        self.condition = condition
        self.message = message

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        metrics = trainer.callback_metrics
        if self.condition(metrics):
            print(self.message)
            trainer.should_stop = True


class RecordAndLogVideoCallback(pl.Callback):
    def __init__(self, every_n_episodes: int = 10):
        self.every_n_episodes = every_n_episodes
        super().__init__()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        # Log only at the *end* of the very first batch of a new episode
        if getattr(pl_module, "episode_steps", 1) != 0:
            return
        ep = getattr(pl_module, "episode", 0)
        if ep == 0 or ep % self.every_n_episodes:
            return

        video_path = self._record_episode(pl_module)
        # ---------------------- consistent logging -----------------------
        logger_run = trainer.logger.experiment      # = wandb run
        logger_run.log(
            {"eval/video": wandb.Video(video_path, fps=30, format="mp4")},
            step=trainer.global_step,               # same counter Lightning uses
        )
        print(f"📹  Logged video for episode {ep} (global step {trainer.global_step})")

    # ---------------------------------------------------------------------
    def _record_episode(self, pl_module) -> str:
        env = gym.make(CONFIG["env_id"], render_mode="rgb_array")
        frames, state = [], env.reset(seed=CONFIG["seed"])[0]
        done, total_reward = False, 0.0

        while not done:
            frames.append(env.render())
            with torch.no_grad():
                #act = pl_module.q_model(torch.as_tensor(state, device=pl_module.device).float()).argmax().item()
                state_t = torch.as_tensor(state, device=pl_module.device).float().unsqueeze(0)  # (1, obs_dim)
                act = pl_module.q_model(state_t).argmax(dim=1).item()
                #act = pl_module.q_model(torch.as_tensor(state, device=pl_module.device).float()).argmax().item()
            state, r, terminated, truncated, _ = env.step(act)
            total_reward += r
            done = terminated or truncated
        env.close()

        # Write mp4
        import tempfile, imageio
        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
            imageio.mimsave(tmp.name, [np.array(f) for f in frames], fps=30)
            return tmp.name

module = DQNModule()
trainer = pl.Trainer(
    logger=wandb_logger,
    log_every_n_steps=50, 
    enable_model_summary=False,
    callbacks=[
        StopOnLambda(
            lambda metrics: metrics.get('reward_mean', -float('inf')) >= CONFIG['target_reward'],
            message=f"Stopping: reward_mean >= {CONFIG['target_reward']}"
        ),
        RecordAndLogVideoCallback(every_n_episodes=10)
    ]
)
trainer.fit(module)

In [ ]:
from tsilva_notebook_utils.video import render_video_from_batches
from PIL import Image

def play(module, n_episodes=1):
    env = gym.make(CONFIG['env_id'], render_mode="rgb_array", width=640, height=480)

    frames = []
    for _ in range(n_episodes):
        total_reward = 0.0
        state, _ = env.reset(seed=CONFIG['seed'])
        done = False
        while not done:
            frame = env.render()
            pil_frame = Image.fromarray(frame)
            frames.append(pil_frame)
            state_t = torch.tensor(state, dtype=torch.float32, device=module.device).unsqueeze(0)
            with torch.no_grad(): action = torch.argmax(module.q_model(state_t), dim=1).item()
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
            state = next_state
        print(f"Episode finished with total reward: {total_reward}")
    env.close()
    print(frames[0])
    return render_video_from_batches(frames)

play(module)